# 🚀 Week 17 & 18: Advanced Deep Learning & Generative AI

**SIT-N Hack-o-Week | 5th Semester (Third Year)**

---

### 📋 Syllabus Overview:
1. **Intro to PyTorch**: Tensors, Autograd dynamic graphs, and custom training loops.
2. **Convolutional Neural Networks (CNNs)**: Convolutions, pooling, spatial dimension calculus, and image classification.
3. **Sequence Modeling**: RNNs, LSTMs, and GRUs (hidden states, gating mechanics, temporal processing).
4. **Regularization in Deep Learning**: Dropout, Batch Normalization, and Early Stopping.
5. **Transfer Learning**: Pretrained CNN backbones (ResNet18 / MobileNet), feature extraction vs. fine-tuning.
6. **Hugging Face Transformers**: Pipelines, tokenizers (BPE / WordPiece), and pretrained NLP models.
7. **LLM APIs & Open Models**: Prompt engineering, OpenAI, Anthropic, Hugging Face Inference API, and Ollama.

---

In [ ]:
# Core Imports & Setup
import os
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

## 1. 🔥 Task 1: Intro to PyTorch (Tensors, Autograd & Training Loop)

PyTorch is an open-source deep learning framework based on the Torch library. Unlike static graph engines, PyTorch utilizes a **Define-by-Run** dynamic computation graph powered by **Autograd**.

In [ ]:
# 1.1 Tensors Creation & Manipulation
t_zeros = torch.zeros(2, 3)
t_rand = torch.randn(3, 3)
t_arange = torch.arange(0, 10, 2)

print("Random Normal Tensor:\n", t_rand)
print("Tensor Shape:", t_rand.shape, "| Dtype:", t_rand.dtype, "| Device:", t_rand.device)

# 1.2 Autograd Mechanics
# Let y = 3x^3 + 2x^2 + 5x + 1. At x = 2.5:
# dy/dx = 9x^2 + 4x + 5 = 9(6.25) + 4(2.5) + 5 = 56.25 + 10 + 5 = 71.25
x = torch.tensor(2.5, requires_grad=True)
y = 3 * (x ** 3) + 2 * (x ** 2) + 5 * x + 1
y.backward()

print(f"Function value at x=2.5: {y.item()}")
print(f"Autograd computed gradient dy/dx: {x.grad.item():.4f} (Expected: 71.25)")

In [ ]:
# 1.3 Custom PyTorch Training Loop on Student Performance Dataset
data_path = os.path.join('..', 'data', 'kaggle_student_performance.csv')
df = pd.read_csv(data_path)
df['Distinction'] = (df['CGPA'] >= 8.5).astype(int)

features = ['StudyHoursPerWeek', 'AttendanceRate', 'MathScore', 'ReadingScore', 'WritingScore']
X_raw = df[features].values
y_raw = df['Distinction'].values

# Standardize
X_norm = (X_raw - X_raw.mean(axis=0)) / (X_raw.std(axis=0) + 1e-7)
X_tensor = torch.tensor(X_norm, dtype=torch.float32)
y_tensor = torch.tensor(y_raw, dtype=torch.float32).unsqueeze(1)

# Neural Network Architecture
class AcademicClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

model = AcademicClassifier(input_dim=5)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Training Loop
history = {'loss': [], 'acc': []}
epochs = 60

for epoch in range(1, epochs + 1):
    model.train()
    optimizer.zero_grad()
    preds = model(X_tensor)
    loss = criterion(preds, y_tensor)
    loss.backward()
    optimizer.step()
    
    acc = ((preds >= 0.5).float() == y_tensor).float().mean().item() * 100
    history['loss'].append(loss.item())
    history['acc'].append(acc)

print(f"Epoch {epochs} - Final Loss: {history['loss'][-1]:.4f} | Accuracy: {history['acc'][-1]:.2f}%")

## 2. 🖼️ Task 2: Convolutional Neural Networks (CNNs)

CNNs are specialized neural networks for grid-like topology data (e.g., 2D images). Key components:
1. **Convolutional Layer**: Applies learnable kernel filters $K \in \mathbb{R}^{C_{in} \times k \times k}$ to extract local spatial features (edges, corners, motifs).
2. **Spatial Dimension Formula**:
$$W_{out} = \left\lfloor \frac{W_{in} - K + 2P}{S} \right\rfloor + 1$$
3. **Pooling Layer**: MaxPool / AvgPool downsamples spatial dimensions while preserving dominant activations.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            # Input: (batch, 1, 28, 28)
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1),  # (16, 28, 28)
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),                                # (16, 14, 14)
            
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1), # (32, 14, 14)
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)                                 # (32, 7, 7)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

cnn_model = SimpleCNN()
sample_image = torch.randn(4, 1, 28, 28)  # Batch of 4 grayscale images
output = cnn_model(sample_image)
print(f"CNN Input Shape: {sample_image.shape}")
print(f"CNN Output Shape: {output.shape} (Batch Size=4, Classes=10)")

## 3. 🔄 Task 3: Sequence Modeling (RNN, LSTM, GRU)

When data has sequential or temporal dependencies (e.g. text, audio, semester-by-semester student trajectories):
- **Vanilla RNN**: $h_t = \tanh(W_{hh} h_{t-1} + W_{xh} x_t + b_h)$. Prone to vanishing gradients ($t > 15$).
- **LSTM (Long Short-Term Memory)**: Solves vanishing gradients via separate cell state $c_t$ and 3 gates (Forget $f_t$, Input $i_t$, Output $o_t$).
- **GRU (Gated Recurrent Unit)**: Streamlines LSTM by merging cell and hidden states with Update ($z_t$) and Reset ($r_t$) gates, using 25% fewer parameters.

In [ ]:
input_size = 8
hidden_size = 32
seq_len = 10
batch_size = 16

sample_seq = torch.randn(batch_size, seq_len, input_size)

# Instantiate all 3 sequence models
rnn = nn.RNN(input_size, hidden_size, batch_first=True)
lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
gru = nn.GRU(input_size, hidden_size, batch_first=True)

out_rnn, h_rnn = rnn(sample_seq)
out_lstm, (h_lstm, c_lstm) = lstm(sample_seq)
out_gru, h_gru = gru(sample_seq)

def count_params(mod):
    return sum(p.numel() for p in mod.parameters())

print(f"RNN Parameters:  {count_params(rnn):<6} | Output Shape: {out_rnn.shape}")
print(f"LSTM Parameters: {count_params(lstm):<6} | Output Shape: {out_lstm.shape}")
print(f"GRU Parameters:  {count_params(gru):<6} | Output Shape: {out_gru.shape}")

## 4. 🛡️ Task 4: Regularization in Deep Learning (Dropout, BatchNorm, Early Stopping)

Deep networks have millions of parameters and easily overfit training data. Three standard defenses:
1. **Dropout**: Randomly deactivates activations during forward passes, breaking co-adaptation.
2. **Batch Normalization**: Rescales mini-batch activations to zero mean, unit variance, stabilizing gradients.
3. **Early Stopping**: Halts training when validation loss fails to decrease for $k$ consecutive epochs.

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float('inf')
        self.counter = 0
        self.early_stop = False

    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

# Demonstration of Regularized Architecture
regularized_block = nn.Sequential(
    nn.Linear(64, 128),
    nn.BatchNorm1d(128),  # Batch Normalization
    nn.ReLU(),
    nn.Dropout(p=0.4),    # Dropout 40%
    nn.Linear(128, 10)
)
print("Regularized PyTorch Module created successfully:", regularized_block)

## 5. 🚀 Task 5: Transfer Learning with Pretrained CNNs

Transfer learning reuses features learned on large datasets (e.g., ImageNet with 1.4 million images):
- **Feature Extraction**: Freeze backbone weights (`param.requires_grad = False`) and train only the final linear head.
- **Fine-Tuning**: Unfreeze the top layers and train at a small learning rate.

In [ ]:
import torchvision.models as models

# Load a lightweight pretrained model
resnet = models.resnet18(weights=None)  # Use architecture

# Freeze all feature extractor layers
for param in resnet.parameters():
    param.requires_grad = False

# Replace classification head for 2 classes (Standard vs Distinction)
num_ftrs = resnet.fc.in_features
resnet.fc = nn.Linear(num_ftrs, 2)

trainable = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in resnet.parameters() if not p.requires_grad)

print(f"Total Parameters:     {frozen + trainable:,}")
print(f"Frozen Backbone:      {frozen:,}")
print(f"Trainable Head:       {trainable:,} (Only 0.009% of parameters require gradient computation!)")

## 6. 🤗 Task 6: Hugging Face Transformers Library Basics

Hugging Face provides unified access to state-of-the-art transformer models (BERT, RoBERTa, T5, LLaMA).
The core pipeline consists of:
$$\text{Raw Text} \xrightarrow{\text{Tokenizer}} \text{Input IDs + Attention Mask} \xrightarrow{\text{Transformer Engine}} \text{Contextual Logits / Predictions}$$

In [ ]:
# Simulated WordPiece Tokenization Pipeline Demonstration
text = "The deep learning curriculum at SIT Nagpur is exceptional!"
words = text.split()
tokens = ['[CLS]'] + [w.lower().strip('!') for w in words] + ['[SEP]']
token_ids = [101] + [1000 + i * 23 for i in range(len(words))] + [102]
attention_mask = [1] * len(token_ids)

token_df = pd.DataFrame({
    'Token': tokens,
    'Token ID': token_ids,
    'Attention Mask': attention_mask
})
print("Transformer Tokenizer Output:")
token_df

## 7. 💬 Task 7: Using LLM APIs (OpenAI, Anthropic, Open Models via Ollama/Hugging Face)

Large Language Models (LLMs) can be queried programmatically via REST APIs or run locally via Ollama.

In [ ]:
# Multi-Provider LLM Client Interface Pattern
class UnifiedLLMClient:
    def __init__(self, provider='simulation', api_key=None):
        self.provider = provider.lower()
        self.api_key = api_key

    def generate(self, prompt, system_instruction="You are a helpful deep learning academic assistant."):
        t0 = time.perf_counter()
        
        # In production with keys:
        # if self.provider == 'openai':
        #     client = OpenAI(api_key=self.api_key)
        #     return client.chat.completions.create(model='gpt-4o-mini', messages=[...])
        
        # Local Academic Fallback Response
        response = (
            f"[Provider: {self.provider.upper()}] Response to '{prompt}':\n"
            "1. PyTorch dynamic computation graphs enable intuitive debugging.\n"
            "2. Pretrained CNNs & Transformers achieve high performance with minimal training data.\n"
            "3. Combining early stopping and dropout ensures strong out-of-distribution generalizability."
        )
        latency = (time.perf_counter() - t0) * 1000
        return {'response': response, 'latency_ms': round(latency, 2)}

client = UnifiedLLMClient(provider='simulation')
result = client.generate("Why use Transfer Learning in computer vision?")
print(result['response'])
print(f"Latency: {result['latency_ms']} ms")

## 8. 🏁 Summary of Key Takeaways

1. **PyTorch Ecosystem**: Tensors + Autograd provide an intuitive, pythonic foundation for neural networks.
2. **Spatial vs Temporal**: CNNs exploit 2D spatial locality via shared kernel weights; RNNs/LSTMs preserve state across temporal sequences.
3. **Regularization Triad**: Dropout, Batch Normalization, and Early Stopping are essential to prevent catastrophic overfitting.
4. **Foundation Models & LLMs**: Pretrained Transformers (Hugging Face) and LLM APIs enable zero-shot / few-shot generalization across high-value multimodal applications.